In [ ]:
import dac
import glob

model_path = dac.utils.download(model_type="16khz")
dac = dac.DAC.load(model_path).eval()

In [ ]:
from nanodrz.data import libritts_test
wav_files = [u.file_url for u in libritts_test()[0].utts]

In [ ]:
import torch
import torchaudio

from nanodrz.utils import play
audio, sr = torchaudio.load(wav_files[0])

print(sr)
resampled_audio = torchaudio.transforms.Resample(sr, 16000)(audio)
print(dac.sample_rate)
play(resampled_audio, 16000)

In [ ]:
resampled_audio = dac.preprocess(resampled_audio, 16000)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(resampled_audio.shape)
x = dac.to(device).encode(audio_data=resampled_audio[None].to(device))

In [ ]:
stacked_code_embs = [dac.quantizer.quantizers[i].codebook(x[1][:,i]) for i in range(len(dac.quantizer.quantizers))]
stacked_code_embs = torch.cat(stacked_code_embs, dim=-1)
stacked_code_embs.shape

In [ ]:
from torch import nn

convcompress = nn.Sequential(
    nn.Conv1d(1024, 1024//2, kernel_size=3),
    nn.GELU(),
    nn.Conv1d(1024//2, 256, kernel_size=3),
    nn.GELU(),
)

convcompress(x[0]).shape

In [ ]:
print("Dac Z Compression", resampled_audio.numel() / x[0].numel())
mel = torchaudio.transforms.MelSpectrogram(n_fft=1024, win_length=256, hop_length=256, n_mels=80)(resampled_audio)
print(mel.shape, audio.shape)
print("Mel Compression", resampled_audio.numel() / mel.numel())

In [ ]:
print(x[0].shape, resampled_audio.shape[-1]/x[0].shape[-1])

In [ ]:
import torch

In [ ]:
resampled_audio = dac.preprocess(torch.rand(1, 1, 16000*3).cuda(), 16000)
dac.cuda().encode(audio_data=resampled_audio)[0].shape

In [ ]:
16000*3/150